# 🎯 [K08_M10] 규정 검색 도우미 완성하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K08/k08_m10_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군사법령·규정 (Level 1) / K08 군 규정 검색 AI - 근거 조항 안내 규정 검색 도우미 구현 |
| **미션** | 미션 10. 규정 검색 도우미 완성하기 (소요시간: 약 2.0시간) |
| **실습 유형** | 규정 텍스트 청킹, 정보 검색(IR) 및 RAG 프롬프트 도우미 구현 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


실행 환경 준비 완료 (Python 3.14.10)


---

## 🎯 미션 10. 규정 검색 도우미 완성하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **전체 기능을 묶은 규정 검색 도우미를 완성하고 자체 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **질문 확장, 검색, 근거 인용 답변, 거절, 질의 기록을 하나의 도우미로 통합하고, 근거 조항 정확도·거절률·처리 시간 등 자체 점검 항목을 모두 통과시킵니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `규정 검색 도우미 코드, 자체 점검 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 통합·완성 — 도우미를 통합하고 자체 점검을 통과했으며 사람의 확인 절차를 명시했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(규정 검색 도우미 코드, 자체 점검 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [11]:
"""규정 구조화, 질문 확장, TF-IDF 검색, 근거 인용 답변, 거절 규칙, RAG 프롬프트 생성을 하나로 묶은 '군 규정 검색 도우미'를 완성하고 자체 점검한다."""
import re
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    rows = []
    for chapter, arts in REGULATIONS:
        for art, text in arts:
            no = int(re.search(r"제(\d+)조", art).group(1))
            title = re.search(r"\((.+)\)", art).group(1)
            rows.append({"no": no, "chapter": chapter, "article": art, "title": title, "text": text})
    return rows


QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

# ---- 공통: 동의어 확장 + TF-IDF 검색기 ----
SYNONYMS = {
    "사진": "촬영", "찍": "촬영", "전기난로": "전열기", "난로": "전열기", "ChatGPT": "외부 AI 서비스 입력",
    "AI": "외부 AI 서비스", "다친": "부상자 환자", "겨울": "동절기", "늦어질": "복귀 지연", "못 받": "연기 추가 일정",
    "고충": "고충 건의 상담", "응시료": "자기개발 지원 응시료", "몇 시": "시간",
}


class Retriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.docs = [c["title"] + " " + c["text"] for c in chunks]
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(self.docs)
        self.D = self.vec.transform(self.docs)

    @staticmethod
    def expand(q):
        return q + " " + " ".join(v for k, v in SYNONYMS.items() if k in q)

    def search(self, q, k=3):
        s = cosine_similarity(self.vec.transform([self.expand(q)]), self.D).ravel()
        return [(self.chunks[i], float(s[i])) for i in s.argsort()[::-1][:k]]


class RegulationAssistant:
    def __init__(self, threshold=0.15):   # 미션 9에서 정한 임계값
        self.R = Retriever(build_chunks())
        self.threshold = threshold
        self.log = []

    def ask(self, q):
        hits = self.R.search(q, 3)
        top, score = hits[0]
        if score < self.threshold:
            ans = "규정에서 관련 조항을 찾지 못했습니다. 행정반 또는 담당 부서에 확인하세요."
            ref = None
        else:
            ans = f"{top['text']} (근거: {top['article']})"
            ref = top["no"]
        self.log.append({"q": q, "ref": ref, "score": round(score, 3)})
        return ans, ref, [c["article"] for c, _ in hits]


A = RegulationAssistant()
for q in ["외출하고 늦게 들어올 것 같아요", "자격증 응시료 지원돼요?", "PX 라면 가격은?"]:
    ans, ref, cand = A.ask(q)
    print(f"Q. {q}\nA. {ans}\n   후보: {', '.join(cand)}\n")

t0 = time.perf_counter()
res = [A.ask(q)[1] for q, _ in QUESTIONS]
ms = (time.perf_counter() - t0) / len(QUESTIONS) * 1000
acc = np.mean([r == a for r, (_, a) in zip(res, QUESTIONS)])
out = ["PX에서 라면 가격은 얼마인가요?", "다음 달 사단장 방문 일정은?", "월급날은 언제인가요?"]
refused = np.mean([A.ask(q)[1] is None for q in out])
checks = {"규정 질문 근거 조항 정확도 80% 이상": acc >= 0.8, "규정 밖 질문 거절 2/3 이상": refused >= 0.66,
          "답변한 질문은 모두 유사도 임계값 이상": all(l["score"] >= A.threshold for l in A.log if l["ref"] is not None),
          "질문당 처리 100ms 이내": ms < 100, "질의 기록 저장": len(A.log) >= 15}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {sum(checks.values())}/{len(checks)} 통과 (정확도 {acc:.0%}, 거절률 {refused:.0%}, 질문당 {ms:.1f} ms)")
print("\n※ 이 도우미는 가상 규정집 학습용이다. 실제 규정 해석과 적용은 지휘계통과 담당 부서의 확인을 따른다.")


Q. 외출하고 늦게 들어올 것 같아요
A. 외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다. (근거: 제9조(외출))
   후보: 제9조(외출), 제10조(복귀 지연), 제11조(정보통신 보안)

Q. 자격증 응시료 지원돼요?
A. 자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다. (근거: 제18조(자기개발 지원))
   후보: 제18조(자기개발 지원), 제13조(외부 AI 서비스), 제15조(환자 발생)

Q. PX 라면 가격은?
A. 규정에서 관련 조항을 찾지 못했습니다. 행정반 또는 담당 부서에 확인하세요.
   후보: 제8조(휴가 조정), 제15조(환자 발생), 제2조(적용 범위)

[PASS] 규정 질문 근거 조항 정확도 80% 이상
[PASS] 규정 밖 질문 거절 2/3 이상
[PASS] 답변한 질문은 모두 유사도 임계값 이상
[PASS] 질문당 처리 100ms 이내
[PASS] 질의 기록 저장
자체 점검 5/5 통과 (정확도 93%, 거절률 100%, 질문당 1.2 ms)

※ 이 도우미는 가상 규정집 학습용이다. 실제 규정 해석과 적용은 지휘계통과 담당 부서의 확인을 따른다.
